# Models Integration with OpenAI, Google Gemini, and Groq
This notebook demonstrates how to load environment keys and initialize different Chat Model providers using LangChain's unified `init_chat_model` utility.

### Step 1: Load Environment Credentials
We load API keys from our `.env` file and set them in the runtime environment.

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

# Load keys into environment variables
os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

### Step 2: Initialize the Chat Model (Using Unified Utility)
Use `init_chat_model` to load the desired provider. If you experience quota or rate limit errors with one provider (e.g. OpenAI), you can easily switch to another provider (e.g. Groq or Gemini) by changing the model name and provider.

In [ ]:
from langchain.chat_models import init_chat_model

# Option A: Groq (using Llama 3)
model = init_chat_model("llama-3.1-8b-instant", model_provider="groq")

# Option B: Google Gemini
# model = init_chat_model("gemini-1.5-flash", model_provider="google_genai")

# Option C: OpenAI
# model = init_chat_model("gpt-4o", model_provider="openai")

model

### Step 2.1: Direct Provider Library Integration (langchain-google-genai)
Instead of using unified wrappers, you can import the model class directly from the provider's integration package. Here, we instantiate Google's Gemini model directly using the `langchain-google-genai` integration library.

**Note:** In version 4.x of the `langchain-google-genai` library, the class is named **`ChatGoogleGenerativeAI`**.

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

# Direct instantiation of ChatGoogleGenerativeAI class
direct_gemini_model = ChatGoogleGenerativeAI(
    model="gemini-1.5-flash",
    temperature=0.7,
    max_tokens=None,
    timeout=None,
    max_retries=2
)

# Invoke the model directly
direct_response = direct_gemini_model.invoke("Say hello in German and French!")
print("Direct Gemini Model Response:")
print(direct_response.content)

### Step 3: Invoke the Model & View Response

In [ ]:
# Send a query to the model
response = model.invoke("Hey! How are you doing?")

### Step 4: Extract Only the Text Response
We can inspect the full response metadata, or extract only the textual content via `.content`.

In [ ]:
# Print only the text output
print("Model Output:")
print(response.content)

# Print raw response metadata
print("\nUsage Metadata:")
print(response.usage_metadata)

### Step 5: Streaming Responses (Advanced Learning)
For longer responses, streaming chunks as they are generated provides a much better user experience.

In [ ]:
print("Streaming Response:")
# Use .stream() to receive chunks in real time
for chunk in model.stream("Write a short 3-line poem about the magic of coding."):
    print(chunk.content, end="", flush=True)

### Step 6: Directing Model Behavior with System Instructions
We can pass a list of messages containing instructions (`SystemMessage`) followed by our query (`HumanMessage`).

In [ ]:
from langchain_core.messages import SystemMessage, HumanMessage

messages = [
    SystemMessage(content="You are a helpful assistant who replies ONLY in a pirate accent."),
    HumanMessage(content="Where can I find the nearest coffee shop?")
]

pirate_response = model.invoke(messages)
print(pirate_response.content)

### Step 7: Structuring Inputs with Prompt Templates (LCEL Chain)
Rather than manually constructing lists of messages, we can use `ChatPromptTemplate` to dynamically format values before passing them to the model.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

# 1. Create a structured prompt template
chef_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a professional chef. Suggest a simple recipe using only the ingredients provided by the user."),
    ("human", "My ingredients are: {ingredients}")
])

# 2. Build a pipeline (chain) using the pipe '|' operator
chef_chain = chef_prompt | model

# 3. Invoke the chain
chef_response = chef_chain.invoke({"ingredients": "rice, eggs, green onions"})
print("Chef Suggestion:")
print(chef_response.content)